In [6]:
#
# Python Module to Generate a
# Sample Financial Data Set
#
# Python for Algorithmic Trading
# (c) Dr. Yves J. Hilpisch
# The Python Quants GmbH
# The following Python script generates sample financial time series data based on a Monte Carlo simulation for a geometric Brownian motion; for more, see Hilpisch (2018, ch. 12):
import numpy as np
import pandas as pd

r = 0.05  # constant short rate
sigma = 0.5  # volatility factor


def generate_sample_data(rows, cols, freq='1min'):
    '''
    Function to generate sample financial data.

    Parameters
    ==========
    rows: int
        number of rows to generate
    cols: int
        number of columns to generate
    freq: str
        frequency string for DatetimeIndex

    Returns
    =======
    df: DataFrame
        DataFrame object with the sample data
    '''
    rows = int(rows)
    cols = int(cols)
    # generate a DatetimeIndex object given the frequency
    index = pd.date_range('2021-1-1', periods=rows, freq=freq)
    # determine time delta in year fractions
    dt = (index[1] - index[0]) / pd.Timedelta(value='365D')
    # print(dt)
    # generate column names
    columns = ['No%d' % i for i in range(cols)]
    # print(columns)
    # generate sample paths for geometric Brownian motion
    raw = np.exp(np.cumsum((r - 0.5 * sigma ** 2) * dt +
                 sigma * np.sqrt(dt) *
                 np.random.standard_normal((rows, cols)), axis=0))
    # print(raw)
    # normalize the data to start at 100
    raw = raw / raw[0] * 100
    # generate the DataFrame object
    df = pd.DataFrame(raw, index=index, columns=columns)
    return df


if __name__ == '__main__':
    rows = 5  # number of rows
    columns = 3  # number of columns
    freq = 'D'  # daily frequency
    print(generate_sample_data(rows, columns, freq))

                   No0         No1         No2
2021-01-01  100.000000  100.000000  100.000000
2021-01-02  101.175044  102.330712  100.998045
2021-01-03  101.301174  103.073769  104.681731
2021-01-04   97.223851  101.470672  104.708326
2021-01-05  100.620087  107.668753  100.826659


In [7]:
print(generate_sample_data(5, 4))

                            No0         No1         No2         No3
2021-01-01 00:00:00  100.000000  100.000000  100.000000  100.000000
2021-01-01 00:01:00  100.070899   99.913594  100.001931  100.008455
2021-01-01 00:02:00  100.104107   99.859286   99.914214  100.037643
2021-01-01 00:03:00  100.139208   99.791377   99.786633  100.082850
2021-01-01 00:04:00  100.094800   99.752252   99.924678  100.142331


In [8]:
%time data = generate_sample_data(rows=5e6, cols=10).round(4)


CPU times: user 1.86 s, sys: 6.71 s, total: 8.57 s
Wall time: 8.6 s


In [9]:
data.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 5000000 entries, 2021-01-01 00:00:00 to 2030-07-05 05:19:00
Freq: min
Data columns (total 10 columns):
 #   Column  Dtype  
---  ------  -----  
 0   No0     float64
 1   No1     float64
 2   No2     float64
 3   No3     float64
 4   No4     float64
 5   No5     float64
 6   No6     float64
 7   No7     float64
 8   No8     float64
 9   No9     float64
dtypes: float64(10)
memory usage: 419.6 MB


In [12]:
h5 = pd.HDFStore('./data.h5', 'w')

In [13]:
%time h5['data']=data

CPU times: user 240 ms, sys: 6.38 s, total: 6.62 s
Wall time: 6.63 s


In [14]:
h5

<class 'pandas.HDFStore'>
File path: ./data.h5

In [15]:
ls -lrtha

total 420M
-rw-r--r-- 1 chenyang chenyang  16K Mar 22 09:26 Chapter3.ipynb
-rw-r--r-- 1 chenyang chenyang  35K Mar 22 09:32 Chapter3_2.ipynb
drwxr-xr-x 6 chenyang chenyang 4.0K Mar 22 10:19 ../
-rw-r--r-- 1 chenyang chenyang 4.7K Mar 22 10:38 Storing_financial_data.ipynb
drwxr-xr-x 2 chenyang chenyang 4.0K Mar 22 10:42 ./
-rw-r--r-- 1 chenyang chenyang 420M Mar 22 10:43 data.h5


In [16]:
h5.close()

In [17]:
h5=pd.HDFStore('data.h5', 'r')

In [18]:
%time data_copy=h5['data']

CPU times: user 293 ms, sys: 868 ms, total: 1.16 s
Wall time: 1.17 s


In [19]:
data_copy.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 5000000 entries, 2021-01-01 00:00:00 to 2030-07-05 05:19:00
Freq: min
Data columns (total 10 columns):
 #   Column  Dtype  
---  ------  -----  
 0   No0     float64
 1   No1     float64
 2   No2     float64
 3   No3     float64
 4   No4     float64
 5   No5     float64
 6   No6     float64
 7   No7     float64
 8   No8     float64
 9   No9     float64
dtypes: float64(10)
memory usage: 419.6 MB


In [20]:
h5.close()

In [21]:
rm -f data.h5

In [24]:
%time data.to_hdf('./data.h5', key='data', format='table')

CPU times: user 1.61 s, sys: 649 ms, total: 2.26 s
Wall time: 2.27 s


In [25]:
ls -lrtha

total 425M
-rw-r--r-- 1 chenyang chenyang  16K Mar 22 09:26 Chapter3.ipynb
-rw-r--r-- 1 chenyang chenyang  35K Mar 22 09:32 Chapter3_2.ipynb
drwxr-xr-x 6 chenyang chenyang 4.0K Mar 22 10:19 ../
-rw-r--r-- 1 chenyang chenyang 4.7K Mar 22 10:38 Storing_financial_data.ipynb
drwxr-xr-x 2 chenyang chenyang 4.0K Mar 22 10:49 ./
-rw-r--r-- 1 chenyang chenyang 425M Mar 22 10:49 data.h5


In [26]:
%time data_copy=pd.read_hdf('data.h5', key='data')

CPU times: user 119 ms, sys: 2.02 s, total: 2.14 s
Wall time: 2.15 s


In [27]:
data_copy.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 5000000 entries, 2021-01-01 00:00:00 to 2030-07-05 05:19:00
Freq: min
Data columns (total 10 columns):
 #   Column  Dtype  
---  ------  -----  
 0   No0     float64
 1   No1     float64
 2   No2     float64
 3   No3     float64
 4   No4     float64
 5   No5     float64
 6   No6     float64
 7   No7     float64
 8   No8     float64
 9   No9     float64
dtypes: float64(10)
memory usage: 419.6 MB


In [28]:
import tables as tb

In [30]:
h5=tb.open_file(filename='data.h5', mode='r')

In [31]:
h5

File(filename=data.h5, title=np.str_(''), mode='r', root_uep='/', filters=Filters(complevel=0, shuffle=False, bitshuffle=False, fletcher32=False, least_significant_digit=None))
/ (RootGroup) np.str_('')
/data (Group) np.str_('')
/data/table (Table(np.int64(5000000),)) np.str_('')
  description := {
  "index": Int64Col(shape=(), dflt=np.int64(0), pos=0),
  "values_block_0": Float64Col(shape=(np.int64(10),), dflt=np.float64(0.0), pos=1)}
  byteorder := 'little'
  chunkshape := (np.int64(2978),)
  autoindex := True
  colindexes := {
    "index": Index(6, mediumshuffle, zlib(1)).is_csi=False}

In [32]:
h5.root.data.table[:3]

array([(1609459200000000, [100.    , 100.    , 100.    , 100.    , 100.    , 100.    , 100.    , 100.    , 100.    , 100.    ]),
       (1609459260000000, [ 99.9615, 100.0892, 100.0079, 100.086 , 100.0819,  99.9517,  99.9926,  99.9863, 100.0184,  99.9736]),
       (1609459320000000, [ 99.8574, 100.1124,  99.9636, 100.1749, 100.039 ,  99.9052, 100.0194, 100.0971,  99.9839,  99.9321])],
      dtype=[('index', '<i8'), ('values_block_0', '<f8', (10,))])

In [33]:
h5.close()

In [34]:
rm data.h5

In [35]:
%time data = generate_sample_data(1e6, 5, '1min').round(4)

CPU times: user 156 ms, sys: 77.3 ms, total: 233 ms
Wall time: 231 ms


In [36]:
data.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 1000000 entries, 2021-01-01 00:00:00 to 2022-11-26 10:39:00
Freq: min
Data columns (total 5 columns):
 #   Column  Non-Null Count    Dtype  
---  ------  --------------    -----  
 0   No0     1000000 non-null  float64
 1   No1     1000000 non-null  float64
 2   No2     1000000 non-null  float64
 3   No3     1000000 non-null  float64
 4   No4     1000000 non-null  float64
dtypes: float64(5)
memory usage: 45.8 MB


In [37]:
import sqlite3 as sq3

In [38]:
con=sq3.connect('data.sql')

In [39]:
%time data.to_sql('data', con)

CPU times: user 2.13 s, sys: 480 ms, total: 2.61 s
Wall time: 2.74 s


1000000

In [40]:
ls -n data.*

-rw-r--r-- 1 1000 1000 105316352 Mar 22 11:07 data.sql


In [41]:
query='SELECT * from data where No1 > 105 and No2 < 108'

In [42]:
%time res = con.execute(query).fetchall()

CPU times: user 402 ms, sys: 77.9 ms, total: 479 ms
Wall time: 479 ms


In [43]:
res[:5]

[('2021-02-22 05:25:00', 136.8156, 110.5838, 107.9348, 112.7958, 83.0776),
 ('2021-02-22 05:26:00', 136.8743, 110.6915, 107.9546, 112.7995, 83.0518),
 ('2021-02-22 05:28:00', 137.0756, 110.5637, 107.9853, 113.0006, 83.2466),
 ('2021-02-22 05:29:00', 137.0621, 110.5985, 107.821, 113.1679, 83.2126),
 ('2021-02-22 05:30:00', 137.0146, 110.5533, 107.927, 113.1226, 83.1862)]

In [ ]:
len(res)

402294

In [45]:
con.close()

In [46]:
ls -lrtha

total 101M
-rw-r--r-- 1 chenyang chenyang  16K Mar 22 09:26 Chapter3.ipynb
-rw-r--r-- 1 chenyang chenyang  35K Mar 22 09:32 Chapter3_2.ipynb
drwxr-xr-x 6 chenyang chenyang 4.0K Mar 22 10:19 ../
-rw-r--r-- 1 chenyang chenyang  15K Mar 22 11:06 Storing_financial_data.ipynb
-rw-r--r-- 1 chenyang chenyang 101M Mar 22 11:07 data.sql
drwxr-xr-x 2 chenyang chenyang 4.0K Mar 22 11:07 ./


In [47]:
rm data.sql